# RAG con Knowledge Graphs · Neo4j + LLM

Cuándo merece la pena un KG RAG vs un vector RAG clásico:

| Caso | Mejor opción |
|---|---|
| Q&A sobre prosa libre, papers, manuales | vector RAG |
| Datos con relaciones complejas (legal, médico, fraude) | **KG RAG** |
| Necesitas razonamiento multi-hop ("X es amigo de Y, Y trabajó con Z, ¿Z conoce a X?") | **KG RAG** |
| Queries con agregaciones ("cuántos", "top N") | **KG RAG** |
| El "qué" importa más que las relaciones | vector RAG |

## Lo que vamos a construir

Un grafo en Neo4j con `Researcher · Article · Topic` y aristas `PUBLISHED · IN_TOPIC`. Sobre él haremos:

1. Q&A por **similitud vectorial** sobre los nodos `Article` (RAG vector clásico, dentro del grafo)
2. Q&A por **traversal Cypher** generado por LLM en lenguaje natural
3. Comparativa de cuándo gana cada uno

![Knowledge Graph](https://raw.githubusercontent.com/dcarpintero/generative-ai-101/main/static/knowledge-graphs.png)

## Setup

Necesitas:

- **Neo4j Aura** (gratis): crea una instancia en https://neo4j.com/product/auradb/ y guarda `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` en un `.env` o exporta como variables de entorno
  - Alternativa local con Docker: `docker run -d -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/<tu-password> -e 'NEO4J_PLUGINS=["apoc"]' neo4j:5` y usa `NEO4J_URI=bolt://localhost:7687`. El plugin **APOC** es obligatorio: `Neo4jGraph` lo usa para leer el esquema (Aura ya lo trae)
- **OpenAI API key** para embeddings y LLM

In [1]:
%pip install -q langchain langchain-core langchain-openai langchain-community langchain-neo4j neo4j python-dotenv

In [2]:
import os, getpass
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
for var in ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

## 1. Conectar a Neo4j

El cliente de `langchain-neo4j` envuelve la conexión y nos deja ejecutar Cypher directamente.

In [3]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)
print("  ✓ conectado a Neo4j")

  ✓ conectado a Neo4j


## 2. Cargar el dataset · `Researcher · Article · Topic`

Dataset sintético de papers académicos: 20 artículos de 7 investigadores, con autor(es), abstract, fecha y topic. Lo cargamos con un `LOAD CSV` que crea los nodos y las aristas en una sola query.

**Esquema**:

- Nodos: `Researcher`, `Article`, `Topic`
- Relaciones: `(Researcher)-[:PUBLISHED]->(Article)`, `(Article)-[:IN_TOPIC]->(Topic)`

In [4]:
load_query = """
LOAD CSV WITH HEADERS
FROM 'https://raw.githubusercontent.com/dcarpintero/generative-ai-101/main/dataset/synthetic_articles.csv'
AS row
FIELDTERMINATOR ';'
MERGE (a:Article {title: row.Title})
SET a.abstract = row.Abstract,
    a.publication_date = date(row.Publication_Date)
FOREACH (researcher IN split(row.Authors, ',') |
    MERGE (p:Researcher {name: trim(researcher)})
    MERGE (p)-[:PUBLISHED]->(a))
FOREACH (topic IN [row.Topic] |
    MERGE (t:Topic {name: trim(topic)})
    MERGE (a)-[:IN_TOPIC]->(t))
"""

graph.query(load_query)
graph.refresh_schema()
print(graph.get_schema)

Node properties:
Article {abstract: STRING, publication_date: DATE, title: STRING}
Researcher {name: STRING}
Topic {name: STRING}
Relationship properties:

The relationships:
(:Article)-[:IN_TOPIC]->(:Topic)
(:Researcher)-[:PUBLISHED]->(:Article)


El grafo está cargado. Si tienes acceso al workspace de Neo4j puedes verlo visualmente.

![Sample 1](https://raw.githubusercontent.com/dcarpintero/generative-ai-101/main/static/kg_sample_00.png)

## 3. Q&A por similitud vectorial · `Neo4jVector`

Primer enfoque: tratar los nodos `Article` como un corpus normal de RAG vector.

`Neo4jVector` calcula embeddings sobre los `text_node_properties` (aquí topic + title + abstract) y los guarda como propiedad del nodo. Las queries se resuelven con búsqueda por similitud coseno **dentro del grafo**.

Esto es útil cuando la pregunta es **textual** ("¿qué papers hablan de X?") y no necesita razonamiento por relaciones.

In [5]:
from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings

vector_index = Neo4jVector.from_existing_graph(
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="articles",
    node_label="Article",
    text_node_properties=["topic", "title", "abstract"],
    embedding_node_property="embedding",
)
print("  ✓ índice vectorial creado sobre nodos Article")

  ✓ índice vectorial creado sobre nodos Article


### Query · cadena LCEL moderna

En la 1ª edición usábamos `RetrievalQA.from_chain_type` (deprecated en Langchain 0.3). La versión 2026 lo monta con LCEL · más legible y componible.

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-5-mini")
retriever = vector_index.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_template(
    "Responde a la pregunta usando los siguientes papers como contexto. "
    "Si los papers no contienen la respuesta, dilo. Cita los títulos.\n\n"
    "Papers:\n{context}\n\nPregunta: {question}"
)

def format_articles(docs):
    return "\n\n".join(f"- {d.page_content}" for d in docs)

vector_chain = (
    {"context": retriever | format_articles, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = vector_chain.invoke("¿Qué papers hablan de cómo la IA podría afectar la vida diaria?")
print(answer)

De los resúmenes proporcionados, estos papers hablan de cómo la IA podría afectar la vida diaria:

- "The Societal Implications of Advanced AI: A Multidisciplinary Analysis", analiza impactos a largo plazo en sociedad, economía y cultura, es decir efectos amplios que inciden en la vida cotidiana.  
- "The Impact of AI on Employment: A Comprehensive Study", estudia efectos de la IA en distintos sectores laborales, lo que afecta directamente el trabajo y la rutina diaria de las personas.  
- "AI and Privacy: Balancing Innovation and Individual Rights", examina la tensión entre avance de la IA y protección de datos personales; la privacidad es un aspecto central de la vida diaria.

El paper "Ethical Considerations in AI Development" no indica en el resumen que trate explícitamente cómo la IA afectará la vida diaria; su foco declarado son las implicaciones éticas y pautas de desarrollo responsable.


## 4. Q&A por traversal Cypher · `GraphCypherQAChain`

Aquí está la magia del KG. Si la pregunta es de **relaciones o agregación**, el vector RAG falla:

- "¿cuántos papers ha publicado Emily Chen?"
- "¿qué pareja de investigadores ha colaborado más?"
- "¿quién tiene más colaboradores distintos?"

Con `GraphCypherQAChain`, el LLM **traduce el lenguaje natural a Cypher**, ejecuta la query en Neo4j, y luego sintetiza la respuesta natural. Es text-to-Cypher con grounding.

⚠️ **Cuidado**: `allow_dangerous_requests=True` es obligatorio porque el LLM genera queries Cypher arbitrarios sobre tu base. En producción limita el acceso del usuario Neo4j a solo lectura.

In [7]:
from langchain_neo4j import GraphCypherQAChain

graph.refresh_schema()

cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=ChatOpenAI(model="gpt-5-mini"),  # genera la Cypher
    qa_llm=ChatOpenAI(model="gpt-5-mini"),       # sintetiza la respuesta natural
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
)

### Demo 1 · contar publicaciones

**Pregunta**: "¿Cuántos artículos ha publicado Emily Chen?"

**Cypher esperada**:

```cypher
MATCH (r:Researcher {name: "Emily Chen"})-[:PUBLISHED]->(a:Article)
RETURN COUNT(a) AS numberOfArticles
```

Respuesta esperada: 7

In [8]:
cypher_chain.invoke({"query": "¿Cuántos artículos ha publicado Emily Chen?"})



> Entering new GraphCypherQAChain chain...


Generated Cypher:
MATCH (r:Researcher {name: "Emily Chen"})-[:PUBLISHED]->(a:Article) RETURN count(a) AS articles_count;
Full Context:
[{'articles_count': 7}]



> Finished chain.


{'query': '¿Cuántos artículos ha publicado Emily Chen?',
 'result': 'Emily Chen ha publicado 7 artículos.'}

### Demo 2 · pareja de colaboradores prolíficos

**Pregunta**: "¿Hay alguna pareja de investigadores que haya publicado más de 3 artículos juntos?"

Esta pregunta es **imposible** para vector RAG · requiere traversal de aristas y agregación. Para KG RAG es trivial.

**Cypher esperada**:

```cypher
MATCH (r1:Researcher)-[:PUBLISHED]->(a:Article)<-[:PUBLISHED]-(r2:Researcher)
WHERE r1 <> r2
WITH r1, r2, COUNT(a) AS sharedArticles
WHERE sharedArticles > 3
RETURN r1.name, r2.name, sharedArticles
```

In [9]:
cypher_chain.invoke(
    {"query": "¿Hay alguna pareja de investigadores que haya publicado más de 3 artículos juntos?"}
)



> Entering new GraphCypherQAChain chain...


Generated Cypher:
MATCH (r1:Researcher)-[:PUBLISHED]->(a:Article)<-[:PUBLISHED]-(r2:Researcher)
WHERE r1.name < r2.name
WITH r1, r2, count(DISTINCT a) AS sharedArticles
WHERE sharedArticles > 3
RETURN r1.name AS researcher1, r2.name AS researcher2, sharedArticles
ORDER BY sharedArticles DESC;
Full Context:
[{'researcher1': 'David Johnson', 'researcher2': 'Emily Chen', 'sharedArticles': 4}, {'researcher1': 'Emily Chen', 'researcher2': 'Robert Taylor', 'sharedArticles': 4}]



> Finished chain.


{'query': '¿Hay alguna pareja de investigadores que haya publicado más de 3 artículos juntos?',
 'result': 'Sí. David Johnson y Emily Chen; Emily Chen y Robert Taylor han publicado 4 artículos juntos.'}

### Demo 3 · investigador con más colaboradores

**Pregunta**: "¿Qué investigador ha colaborado con más peers distintos?"

**Cypher esperada**:

```cypher
MATCH (r:Researcher)-[:PUBLISHED]->(:Article)<-[:PUBLISHED]-(peer:Researcher)
WITH r, COUNT(DISTINCT peer) AS peerCount
RETURN r.name AS researcher, peerCount
ORDER BY peerCount DESC
LIMIT 1
```

Ojo con los empates: en este dataset **David Johnson, Michael Brown y Sarah Lee** tienen 6 colaboradores distintos cada uno. Con `LIMIT 1` la respuesta es uno de los tres según el orden en que Neo4j devuelva las filas; una Cypher sin `LIMIT` (o con `LIMIT 3`) muestra el empate. Es un buen ejemplo de por qué conviene revisar la Cypher generada y no solo la respuesta.

In [10]:
cypher_chain.invoke({"query": "¿Qué investigador ha colaborado con más peers distintos?"})



> Entering new GraphCypherQAChain chain...


Generated Cypher:
MATCH (r:Researcher)-[:PUBLISHED]->(a:Article)<-[:PUBLISHED]-(p:Researcher)
WHERE r <> p
WITH r, COUNT(DISTINCT p) AS peerCount
RETURN r.name AS researcher, peerCount
ORDER BY peerCount DESC
LIMIT 1
Full Context:
[{'researcher': 'David Johnson', 'peerCount': 6}]



> Finished chain.


{'query': '¿Qué investigador ha colaborado con más peers distintos?',
 'result': 'David Johnson ha colaborado con más peers distintos: 6.'}

## 5. Híbrido · vector + Cypher · lo mejor de ambos

En producción rara vez usas uno solo. El patrón habitual:

1. **Vector RAG** para encontrar los nodos relevantes ("papers que hablan de X")
2. **Cypher** sobre esos nodos para extraer relaciones ("y los autores que más los citan")

Hay frameworks que orquestan esto automáticamente:

- **GraphRAG** (Microsoft, 2024) · construye comunidades y resúmenes jerárquicos
- **LightRAG** (2024) · híbrido más ligero, open-source
- **Neo4j GraphRAG** package · oficial, integra ambos modos

## Cierre

Lo que has visto:

- El **mismo grafo** soporta dos modos de Q&A: vector (textual) y Cypher (relacional)
- `Neo4jVector` te da RAG clásico **dentro del grafo**, sin DB separada
- `GraphCypherQAChain` traduce lenguaje natural a Cypher · text-to-query con grounding
- Las preguntas con **agregaciones o multi-hop** son donde KG brilla y vector RAG no llega

**¿Cuándo invertir en montar un KG?**

- Si tu dominio tiene relaciones explícitas que importan al usuario (legal, biotech, fraude, supply chain) · **sí**
- Si es prosa libre y las preguntas son textuales ("qué dice el manual sobre X") · **no, vector RAG es más rápido y barato**
- En híbrido, casi siempre · usa lo que necesite cada query